# 01. Data Ingestion, Validation, Cleaning, and H3 Preparation  
# 01. 数据导入、验证、清洗与 H3 准备

**Pipeline position / 主线位置:** raw 2022-2024 CSV files -> MatrixOne staging -> typed clean table -> H3 analysis table -> fixed train/test split.  
**流程位置：** 2022-2024 原始 CSV -> MatrixOne 暂存表 -> 类型化清洗表 -> H3 分析表 -> 固定训练测试切分。

This notebook is the entry point for the project. It checks the raw files when they are available, validates the tables already built in MatrixOne, measures cleaning quality, verifies H3 coverage, and records the split used by every later experiment. Large row-level work stays in MatrixOne; Python only reads headers and summary tables.  
本 Notebook 是项目入口。它会在原始文件可用时检查 CSV，并验证 MatrixOne 中已经建好的表、清洗质量、H3 覆盖率以及后续实验统一使用的数据切分。大规模逐行处理留在 MatrixOne，Python 只读取 header 和汇总结果。

**Inputs / 输入:** `tnp_2022_full.csv`, `tnp_2023_2024_full.csv`, or the existing MatrixOne tables.  
**Outputs / 输出:** validation tables, schema comparison, data-quality report, and split manifest.

## 1. Environment / 环境

In [ ]:
# Cell 1 - Imports and portable configuration / 导入包与可移植配置
from pathlib import Path
from getpass import getpass
import csv
import json
import os
import re
import warnings

import pandas as pd
import pymysql

PROJECT_DIR = Path(
    os.getenv("CHICAGO_TNP_PROJECT_DIR", str(Path.cwd()))
).expanduser().resolve()
RAW_DIR = Path(
    os.getenv("CHICAGO_TNP_RAW_DIR", str(PROJECT_DIR / "raw" / "full"))
).expanduser().resolve()
OUTPUT_DIR = PROJECT_DIR / "notebook_outputs_data_preparation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    "2022": RAW_DIR / "tnp_2022_full.csv",
    "2023_2024": RAW_DIR / "tnp_2023_2024_full.csv",
}

MO_HOST = os.getenv("MATRIXONE_HOST", "127.0.0.1")
MO_PORT = int(os.getenv("MATRIXONE_PORT", "6001"))
MO_USER = os.getenv("MATRIXONE_USER", "root")
MO_PASSWORD = os.getenv("MATRIXONE_PASSWORD") or getpass(
    "MatrixOne password / MatrixOne 密码: "
)
MO_DB = os.getenv("MATRIXONE_DATABASE", "chicago_tnp")

STAGING_TABLES = ["tnp_2022_raw_staging", "tnp_2023_2024_raw_staging"]
CLEAN_TABLE = "unified_trips_clean"
H3_TABLE = "unified_trips_h3_res9"
TRAIN_START = "2022-01-01"
TRAIN_END_EXCLUSIVE = "2024-01-01"
TEST_END_EXCLUSIVE = "2025-01-01"
RUN_FULL_LOAD = False
BUILD_H3_TABLE = False

print("PROJECT_DIR / 项目目录:", PROJECT_DIR)
print("RAW_DIR / 原始数据目录:", RAW_DIR)
print("OUTPUT_DIR / 输出目录:", OUTPUT_DIR)

## 2. Raw-file validation / 原始文件验证

In [ ]:
# Cell 2 - Inspect raw CSV headers and sample rows / 检查原始 CSV 的 header 和样例
def read_csv_header(path):
    """Read one CSV header without loading the file. / 只读取 CSV header，不加载整份文件。"""
    with path.open("r", encoding="utf-8-sig", errors="replace", newline="") as handle:
        return next(csv.reader(handle))


def snake_case(name):
    """Convert a raw column name to a SQL-safe name. / 把原始列名转成 SQL 安全列名。"""
    value = name.strip().lower().replace("%", "percent").replace("#", "number")
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return re.sub(r"_+", "_", value).strip("_")


raw_headers = {}
raw_file_rows = []
for label, path in RAW_FILES.items():
    row = {"source": label, "path": str(path), "exists": path.exists()}
    if path.exists():
        header = read_csv_header(path)
        clean_header = [snake_case(col) for col in header]
        if len(clean_header) != len(set(clean_header)):
            raise RuntimeError(f"Duplicate cleaned columns in {label}. / {label} 清理后列名重复。")
        raw_headers[label] = clean_header
        row.update({"size_gb": path.stat().st_size / 1024**3, "columns": len(header)})
    raw_file_rows.append(row)

raw_file_check = pd.DataFrame(raw_file_rows)
display(raw_file_check)

if len(raw_headers) == len(RAW_FILES):
    h22 = set(raw_headers["2022"])
    h2324 = set(raw_headers["2023_2024"])
    schema_difference = pd.DataFrame({
        "2023_2024_only": pd.Series(sorted(h2324 - h22), dtype="object"),
        "2022_only": pd.Series(sorted(h22 - h2324), dtype="object"),
    })
    display(schema_difference)
    schema_difference.to_csv(OUTPUT_DIR / "raw_schema_difference.csv", index=False)
else:
    print("Raw CSV files are not mounted; MatrixOne validation will continue. / 未挂载原始 CSV，继续验证 MatrixOne。")

The source schemas are expected to differ by three 2023-2024 fields: `percent_time_chicago`, `percent_distance_chicago`, and `shared_trip_match`. During unification, the 2022 rows receive `NULL` for fields that do not exist in that source. The source timestamp formats are normalized before entering the typed clean table.  
原始 schema 预计有三个只出现在 2023-2024 的字段：`percent_time_chicago`、`percent_distance_chicago` 和 `shared_trip_match`。合并时，2022 不存在的字段填 `NULL`。两份源文件的时间格式在进入类型化清洗表前统一。

## 3. Reproducible staging and cleaning SQL / 可复现的暂存与清洗 SQL

In [ ]:
# Cell 3 - Build the guarded full-load SQL / 生成受开关保护的全量导入 SQL
DEFAULT_HEADERS = {
    "2022": [
        "trip_id", "trip_start_timestamp", "trip_end_timestamp", "trip_seconds",
        "trip_miles", "pickup_census_tract", "dropoff_census_tract",
        "pickup_community_area", "dropoff_community_area", "fare", "tip",
        "additional_charges", "trip_total", "shared_trip_authorized", "trips_pooled",
        "pickup_centroid_latitude", "pickup_centroid_longitude", "pickup_centroid_location",
        "dropoff_centroid_latitude", "dropoff_centroid_longitude", "dropoff_centroid_location",
    ],
    "2023_2024": [
        "trip_id", "trip_start_timestamp", "trip_end_timestamp", "trip_seconds",
        "trip_miles", "percent_time_chicago", "percent_distance_chicago",
        "pickup_census_tract", "dropoff_census_tract", "pickup_community_area",
        "dropoff_community_area", "fare", "tip", "additional_charges", "trip_total",
        "shared_trip_authorized", "shared_trip_match", "trips_pooled",
        "pickup_centroid_latitude", "pickup_centroid_longitude", "pickup_centroid_location",
        "dropoff_centroid_latitude", "dropoff_centroid_longitude", "dropoff_centroid_location",
    ],
}


def quote_ident(name):
    """Quote a SQL identifier. / 给 SQL 标识符加引号。"""
    return "`" + name.replace("`", "``") + "`"


def quote_string(value):
    """Quote a SQL string value. / 给 SQL 字符串值加引号。"""
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def staging_sql(table_name, columns, file_path):
    """Build staging DDL and LOAD DATA SQL. / 生成暂存表建表与导入 SQL。"""
    column_sql = ",\n".join(f"  {quote_ident(col)} VARCHAR(1024)" for col in columns)
    return f"""DROP TABLE IF EXISTS {quote_ident(table_name)};
CREATE TABLE {quote_ident(table_name)} (
{column_sql}
);
LOAD DATA LOCAL INFILE {quote_string(file_path)}
INTO TABLE {quote_ident(table_name)}
FIELDS TERMINATED BY ','
OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\\n'
IGNORE 1 LINES;"""


headers_for_sql = {
    label: raw_headers.get(label, DEFAULT_HEADERS[label])
    for label in DEFAULT_HEADERS
}

typed_columns = """
  `source_file` VARCHAR(20),
  `trip_id` VARCHAR(80),
  `trip_start_timestamp` DATETIME,
  `trip_end_timestamp` DATETIME,
  `trip_seconds` BIGINT,
  `trip_miles` DOUBLE,
  `pickup_community_area` INT,
  `dropoff_community_area` INT,
  `fare` DOUBLE,
  `tip` DOUBLE,
  `additional_charges` DOUBLE,
  `trip_total` DOUBLE,
  `shared_trip_authorized` TINYINT,
  `shared_trip_match` TINYINT,
  `trips_pooled` INT,
  `pickup_centroid_latitude` DOUBLE,
  `pickup_centroid_longitude` DOUBLE,
  `pickup_centroid_location` VARCHAR(120),
  `dropoff_centroid_latitude` DOUBLE,
  `dropoff_centroid_longitude` DOUBLE,
  `dropoff_centroid_location` VARCHAR(120)
""".strip()


def typed_select(source_label, table_name, time_format, has_match):
    """Build one typed SELECT for the unified clean table. / 为统一清洗表生成类型转换 SELECT。"""
    match_expr = "CAST(NULLIF(TRIM(shared_trip_match), '') AS TINYINT)" if has_match else "NULL"
    return f"""
INSERT INTO {quote_ident(CLEAN_TABLE)}
SELECT
  {quote_string(source_label)},
  NULLIF(TRIM(trip_id), ''),
  STR_TO_DATE(NULLIF(TRIM(trip_start_timestamp), ''), {quote_string(time_format)}),
  STR_TO_DATE(NULLIF(TRIM(trip_end_timestamp), ''), {quote_string(time_format)}),
  CAST(NULLIF(TRIM(trip_seconds), '') AS BIGINT),
  CAST(NULLIF(TRIM(trip_miles), '') AS DOUBLE),
  CAST(NULLIF(TRIM(pickup_community_area), '') AS INT),
  CAST(NULLIF(TRIM(dropoff_community_area), '') AS INT),
  CAST(NULLIF(TRIM(fare), '') AS DOUBLE),
  CAST(NULLIF(TRIM(tip), '') AS DOUBLE),
  CAST(NULLIF(TRIM(additional_charges), '') AS DOUBLE),
  CAST(NULLIF(TRIM(trip_total), '') AS DOUBLE),
  CAST(NULLIF(TRIM(shared_trip_authorized), '') AS TINYINT),
  {match_expr},
  CAST(NULLIF(TRIM(trips_pooled), '') AS INT),
  CAST(NULLIF(TRIM(pickup_centroid_latitude), '') AS DOUBLE),
  CAST(NULLIF(TRIM(pickup_centroid_longitude), '') AS DOUBLE),
  NULLIF(TRIM(pickup_centroid_location), ''),
  CAST(NULLIF(TRIM(dropoff_centroid_latitude), '') AS DOUBLE),
  CAST(NULLIF(TRIM(dropoff_centroid_longitude), '') AS DOUBLE),
  NULLIF(TRIM(dropoff_centroid_location), '')
FROM {quote_ident(table_name)};
""".strip()


full_load_sql = "\n\n".join([
    f"CREATE DATABASE IF NOT EXISTS {quote_ident(MO_DB)};",
    f"USE {quote_ident(MO_DB)};",
    staging_sql(STAGING_TABLES[0], headers_for_sql["2022"], RAW_FILES["2022"]),
    staging_sql(STAGING_TABLES[1], headers_for_sql["2023_2024"], RAW_FILES["2023_2024"]),
    f"DROP TABLE IF EXISTS {quote_ident(CLEAN_TABLE)};\nCREATE TABLE {quote_ident(CLEAN_TABLE)} (\n{typed_columns}\n);",
    typed_select("2022", STAGING_TABLES[0], "%Y-%m-%dT%H:%i:%s.%f", False),
    typed_select("2023_2024", STAGING_TABLES[1], "%m/%d/%Y %h:%i:%s %p", True),
])

full_load_sql_path = OUTPUT_DIR / "01_full_staging_and_clean_load.sql"
full_load_sql_path.write_text(full_load_sql + "\n", encoding="utf-8")
print("Generated SQL / 已生成 SQL:", full_load_sql_path)
print("RUN_FULL_LOAD / 是否执行全量导入:", RUN_FULL_LOAD)

if RUN_FULL_LOAD:
    missing_raw = [str(path) for path in RAW_FILES.values() if not path.exists()]
    if missing_raw:
        raise FileNotFoundError(f"Missing raw files / 缺少原始文件: {missing_raw}")
    print("The next cell will run the guarded full load. / 下一 Cell 将执行受保护的全量导入。")

## 4. MatrixOne connection and table lineage / MatrixOne 连接与表关系

In [ ]:
# Cell 4 - Connect and define small SQL helpers / 连接并定义 SQL 工具
def new_connection():
    """Open a MatrixOne connection. / 创建 MatrixOne 连接。"""
    return pymysql.connect(
        host=MO_HOST,
        port=MO_PORT,
        user=MO_USER,
        password=MO_PASSWORD,
        database=MO_DB,
        charset="utf8mb4",
        autocommit=True,
        read_timeout=1800,
        write_timeout=1800,
        local_infile=True,
    )


conn = new_connection()

if RUN_FULL_LOAD:
    statements = [statement.strip() for statement in full_load_sql.split(";\n") if statement.strip()]
    with conn.cursor() as cursor:
        for position, statement in enumerate(statements, start=1):
            print(f"Running full-load statement {position}/{len(statements)}")
            cursor.execute(statement)
    print("Full staging and clean load finished. / 全量暂存与清洗导入完成。")


def query_df(sql, params=None, retry=True):
    """Run SQL and return a DataFrame. / 执行 SQL 并返回 DataFrame。"""
    global conn
    try:
        conn.ping(reconnect=True)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return pd.read_sql_query(sql, conn, params=params)
    except pymysql.err.OperationalError:
        if not retry:
            raise
        conn.close()
        conn = new_connection()
        return query_df(sql, params=params, retry=False)


def table_exists(table_name):
    """Check whether a MatrixOne table exists. / 检查 MatrixOne 表是否存在。"""
    sql = """
    SELECT COUNT(*) AS n
    FROM information_schema.tables
    WHERE table_schema = %s AND table_name = %s
    """
    return int(query_df(sql, [MO_DB, table_name]).loc[0, "n"]) > 0


table_inventory = []
for table_name in [*STAGING_TABLES, CLEAN_TABLE, H3_TABLE]:
    table_inventory.append({"table": table_name, "exists": table_exists(table_name)})
table_inventory = pd.DataFrame(table_inventory)
display(table_inventory)

## 5. Clean-table validation / 清洗表验证

In [ ]:
# Cell 5 - Validate row counts, types, and cleaning rules / 验证行数、类型和清洗规则
required_tables = [CLEAN_TABLE] + ([] if BUILD_H3_TABLE else [H3_TABLE])
for required_table in required_tables:
    if not table_exists(required_table):
        raise RuntimeError(f"Missing table: {required_table}. / 缺少表：{required_table}。")

clean_schema = query_df(f"DESCRIBE {CLEAN_TABLE};")
required_clean_columns = {
    "source_file", "trip_start_timestamp", "trip_end_timestamp",
    "trip_seconds", "trip_miles", "fare", "tip",
    "pickup_centroid_location", "dropoff_centroid_location",
}
actual_clean_columns = set(clean_schema["Field"].astype(str))
missing_clean_columns = sorted(required_clean_columns - actual_clean_columns)
if missing_clean_columns:
    raise RuntimeError(f"Missing clean columns / 清洗表缺少字段: {missing_clean_columns}")

row_counts = query_df(f"""
SELECT source_file, COUNT(*) AS rows_n,
       MIN(trip_start_timestamp) AS min_start,
       MAX(trip_start_timestamp) AS max_start
FROM {CLEAN_TABLE}
GROUP BY source_file
ORDER BY source_file;
""")

quality_summary = query_df(f"""
SELECT
    COUNT(*) AS rows_n,
    SUM(CASE WHEN trip_start_timestamp IS NULL THEN 1 ELSE 0 END) AS null_start,
    SUM(CASE WHEN trip_end_timestamp IS NULL THEN 1 ELSE 0 END) AS null_end,
    SUM(CASE WHEN trip_seconds < 0 THEN 1 ELSE 0 END) AS negative_seconds,
    SUM(CASE WHEN trip_miles < 0 THEN 1 ELSE 0 END) AS negative_miles,
    SUM(CASE WHEN fare < 0 THEN 1 ELSE 0 END) AS negative_fare,
    SUM(CASE WHEN tip < 0 THEN 1 ELSE 0 END) AS negative_tip
FROM {CLEAN_TABLE};
""")

display(row_counts)
display(clean_schema)
display(quality_summary)
row_counts.to_csv(OUTPUT_DIR / "clean_row_counts.csv", index=False)
quality_summary.to_csv(OUTPUT_DIR / "clean_quality_summary.csv", index=False)

The clean table is the shared contract for later notebooks. Timestamps are stored as `DATETIME`; numeric fields are stored as numeric types; source-only fields stay nullable. Zero-valued trips are kept because they may be real data-quality signals, while impossible negative values are reported explicitly.  
清洗表是后续 Notebook 共用的数据约定。时间存为 `DATETIME`，数值字段存为数值类型，源文件缺失字段允许为 `NULL`。数值为零的行程可能是有意义的数据质量信号，因此保留；不合理的负数则单独统计。

## 6. H3 coverage and spatial contract / H3 覆盖与空间约定

In [ ]:
# Cell 6 - Build or validate H3 rows and location coverage / 构建或验证 H3 行数与地点覆盖
if BUILD_H3_TABLE:
    try:
        import h3
    except ImportError as error:
        raise ImportError("Install h3 before building the H3 table: pip install h3") from error

    centroid_rows = query_df(f"""
    SELECT pickup_centroid_location AS location,
           pickup_centroid_latitude AS latitude,
           pickup_centroid_longitude AS longitude
    FROM {CLEAN_TABLE}
    WHERE pickup_centroid_location IS NOT NULL
      AND pickup_centroid_latitude IS NOT NULL
      AND pickup_centroid_longitude IS NOT NULL
    UNION
    SELECT dropoff_centroid_location AS location,
           dropoff_centroid_latitude AS latitude,
           dropoff_centroid_longitude AS longitude
    FROM {CLEAN_TABLE}
    WHERE dropoff_centroid_location IS NOT NULL
      AND dropoff_centroid_latitude IS NOT NULL
      AND dropoff_centroid_longitude IS NOT NULL;
    """)

    def to_h3(latitude, longitude, resolution=9):
        """Convert one coordinate to an H3 cell. / 把一个坐标转换成 H3 单元。"""
        if hasattr(h3, "latlng_to_cell"):
            return h3.latlng_to_cell(float(latitude), float(longitude), resolution)
        return h3.geo_to_h3(float(latitude), float(longitude), resolution)

    centroid_rows = centroid_rows.drop_duplicates("location").copy()
    centroid_rows["h3"] = [
        to_h3(lat, lon) for lat, lon in zip(centroid_rows["latitude"], centroid_rows["longitude"])
    ]
    centroid_rows.to_csv(OUTPUT_DIR / "centroid_h3_map_res9.csv", index=False)

    with conn.cursor() as cursor:
        cursor.execute("DROP TABLE IF EXISTS centroid_h3_map_res9")
        cursor.execute("""
        CREATE TABLE centroid_h3_map_res9 (
            location VARCHAR(120), latitude DOUBLE, longitude DOUBLE, h3 VARCHAR(32)
        )
        """)
        cursor.executemany(
            "INSERT INTO centroid_h3_map_res9 VALUES (%s, %s, %s, %s)",
            list(centroid_rows[["location", "latitude", "longitude", "h3"]].itertuples(index=False, name=None)),
        )
        cursor.execute(f"DROP TABLE IF EXISTS {H3_TABLE}")
        cursor.execute(f"""
        CREATE TABLE {H3_TABLE} AS
        SELECT c.*, pickup_map.h3 AS pickup_h3, dropoff_map.h3 AS dropoff_h3
        FROM {CLEAN_TABLE} c
        LEFT JOIN centroid_h3_map_res9 pickup_map
          ON c.pickup_centroid_location = pickup_map.location
        LEFT JOIN centroid_h3_map_res9 dropoff_map
          ON c.dropoff_centroid_location = dropoff_map.location
        """)
    print("H3 table built. / H3 表已构建。")

h3_schema = query_df(f"DESCRIBE {H3_TABLE};")
h3_columns = set(h3_schema["Field"].astype(str))
required_h3_columns = {"pickup_h3", "dropoff_h3"}
missing_h3_columns = sorted(required_h3_columns - h3_columns)
if missing_h3_columns:
    raise RuntimeError(f"Missing H3 columns / 缺少 H3 字段: {missing_h3_columns}")

h3_summary = query_df(f"""
SELECT
    COUNT(*) AS rows_h3,
    SUM(CASE WHEN pickup_h3 IS NULL THEN 1 ELSE 0 END) AS null_pickup_h3,
    SUM(CASE WHEN dropoff_h3 IS NULL THEN 1 ELSE 0 END) AS null_dropoff_h3,
    COUNT(DISTINCT pickup_h3) AS distinct_pickup_h3,
    COUNT(DISTINCT dropoff_h3) AS distinct_dropoff_h3
FROM {H3_TABLE};
""")

source_h3_summary = query_df(f"""
SELECT source_file,
       COUNT(*) AS rows_n,
       SUM(CASE WHEN pickup_h3 IS NULL THEN 1 ELSE 0 END) AS null_pickup_h3,
       SUM(CASE WHEN dropoff_h3 IS NULL THEN 1 ELSE 0 END) AS null_dropoff_h3,
       COUNT(DISTINCT pickup_h3) AS distinct_pickup_h3,
       COUNT(DISTINCT dropoff_h3) AS distinct_dropoff_h3
FROM {H3_TABLE}
GROUP BY source_file
ORDER BY source_file;
""")

display(h3_summary)
display(source_h3_summary)
h3_summary.to_csv(OUTPUT_DIR / "h3_coverage_summary.csv", index=False)
source_h3_summary.to_csv(OUTPUT_DIR / "h3_coverage_by_source.csv", index=False)

## 7. Shared time split / 统一时间切分

In [ ]:
# Cell 7 - Record the split used by later experiments / 记录后续实验统一使用的切分
split_counts = query_df(f"""
SELECT
    CASE
        WHEN trip_start_timestamp >= '{TRAIN_START}'
         AND trip_start_timestamp < '{TRAIN_END_EXCLUSIVE}' THEN 'train_2022_2023'
        WHEN trip_start_timestamp >= '{TRAIN_END_EXCLUSIVE}'
         AND trip_start_timestamp < '{TEST_END_EXCLUSIVE}' THEN 'test_2024'
        ELSE 'outside_scope'
    END AS split_name,
    COUNT(*) AS rows_n
FROM {H3_TABLE}
GROUP BY split_name
ORDER BY split_name;
""")

split_manifest = {
    "source_table": H3_TABLE,
    "train_start": TRAIN_START,
    "train_end_exclusive": TRAIN_END_EXCLUSIVE,
    "test_end_exclusive": TEST_END_EXCLUSIVE,
    "rule": "Train on 2022-2023 and evaluate on 2024; do not use random row splits.",
}
(OUTPUT_DIR / "split_manifest.json").write_text(
    json.dumps(split_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
display(split_counts)
print(json.dumps(split_manifest, indent=2, ensure_ascii=False))

## Result snapshot / 结果快照

- The validated full clean and H3 tables contain 243,479,296 rows: 69,109,780 from 2022 and 174,369,516 from 2023-2024. / 已验证的全量清洗表与 H3 表共有 243,479,296 行，其中 2022 为 69,109,780 行，2023-2024 为 174,369,516 行。
- The location mapping contains 879 source centroids and 858 distinct H3 resolution-9 cells. / 地点映射包含 879 个源 centroid，对应 858 个不同的 H3 resolution-9 单元。
- Every later predictive test uses 2022-2023 for training and 2024 for evaluation. / 后续所有预测实验统一使用 2022-2023 训练、2024 测试。

**Next / 下一步:** Notebook 02 aggregates the validated H3 table into hourly demand and builds the first forecasting models.

In [ ]:
# Cell 8 - Close the database connection / 关闭数据库连接
conn.close()
print("MatrixOne connection closed. / MatrixOne 连接已关闭。")